# Argmin cifrato lato server su GPU (Colab)

Misura il tempo dell'**argmin omomorfico** (la parte che decide il match 1:N senza che il server veda nulla) su **GPU NVIDIA T4**, e lo confronta con la CPU della stessa macchina.

Il tempo dipende solo da `(N, dim, bit-width)`, non dagli embedding reali: quindi usiamo una galleria sintetica quantizzata che riproduce lo stesso bit-width del caso reale (dim 128 -> ~11 bit, il punto a ~90% di accuratezza).

## Prima di lanciare
**Runtime -> Cambia tipo di runtime -> Acceleratore hardware: GPU (T4)**. Poi **Runtime -> Esegui tutto**.

Quando finisce, copia l'output dell'ultima cella e incollamelo.

In [ ]:
# 1) Verifica che la GPU sia una T4 (o comunque sm_70+: T4=7.5, V100=7.0, L4/A100 ok)
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

In [ ]:
# 2) Installa concrete-python con supporto GPU (wheel dall'indice GPU di Zama)
!pip install -q concrete-python --extra-index-url https://pypi.zama.ai/gpu
import concrete, sys
print('concrete-python', concrete.__version__ if hasattr(concrete,'__version__') else '?')

In [ ]:
# 3) Sanity check: la GPU e' davvero usabile da Concrete?
from concrete import fhe
import numpy as np

@fhe.compiler({'x': 'encrypted'})
def _f(x):
    return x + 1

try:
    _c = _f.compile(range(64), fhe.Configuration(use_gpu=True))
    _c.keygen()
    print('GPU usabile da Concrete:', int(_c.decrypt(_c.run(_c.encrypt(7)))) == 8)
except Exception as e:
    print('PROBLEMA GPU:', repr(e))
    print('-> il wheel potrebbe non avere supporto GPU, oppure il runtime non e\' su GPU.')

In [ ]:
# 4) Benchmark argmin cifrato: GPU vs CPU
import time, numpy as np
from concrete import fhe

def costruisci(N, DIM, Q, use_gpu, seed=0):
    rng = np.random.RandomState(seed)
    G = rng.randint(-Q, Q + 1, size=(N, DIM)).astype(np.int64)   # galleria in chiaro (server)
    b_sq = np.sum(G * G, axis=1).astype(np.int64)                # ||b||^2 precomputato in chiaro

    def argmin_server(a):
        # distanza espansa: ||a-b||^2 = ||b||^2 - 2 a.b   (||a||^2 e' costante -> irrilevante per argmin)
        p = b_sq - 2 * (G @ a)
        idx = fhe.zeros(())
        val = p[0]
        for i in range(1, N):
            lt = (p[i] < val).astype(np.int64)
            idx = lt * i + (1 - lt) * idx
            val = np.minimum(val, p[i])
        return idx

    # inputset rappresentativo: tanti probe casuali, cosi' Concrete inferisce il bit-width giusto
    inputset = [rng.randint(-Q, Q + 1, size=DIM).astype(np.int64) for _ in range(200)]
    cfg = fhe.Configuration(
        comparison_strategy_preference=[fhe.ComparisonStrategy.CHUNKED],
        min_max_strategy_preference=[fhe.MinMaxStrategy.CHUNKED],
        use_gpu=use_gpu,
    )
    compiler = fhe.Compiler(argmin_server, {'a': 'encrypted'})
    t0 = time.time()
    circuit = compiler.compile(inputset, cfg)
    return circuit, G, b_sq, time.time() - t0

def misura(N, DIM, Q, use_gpu):
    dev = 'GPU' if use_gpu else 'CPU'
    try:
        circuit, G, b_sq, t_comp = costruisci(N, DIM, Q, use_gpu)
    except Exception as e:
        print(f'[{dev}] N={N} dim={DIM}: COMPILE FALLITA -> {repr(e)[:120]}')
        return
    try:
        bw = circuit.graph.maximum_integer_bit_width()
    except Exception:
        bw = '?'
    circuit.keygen()
    rng = np.random.RandomState(123)
    a = rng.randint(-Q, Q + 1, size=DIM).astype(np.int64)
    enc = circuit.encrypt(a)
    t0 = time.time(); res = circuit.run(enc); t_run = time.time() - t0
    out = int(circuit.decrypt(res))
    atteso = int(np.argmin(b_sq - 2 * (G @ a)))
    ok = 'OK' if out == atteso else 'X'
    print(f'[{dev}] N={N} dim={DIM} bit={bw}: compile={t_comp:.1f}s run={t_run:.2f}s  pred={out} atteso={atteso} {ok}')

print('=== GPU (la macchina T4) ===')
for DIM in [64, 128, 256]:
    misura(N=8, DIM=DIM, Q=2, use_gpu=True)

print('\n=== CPU (baseline, stessa macchina, solo dim 128) ===')
misura(N=8, DIM=128, Q=2, use_gpu=False)

print('\nIl numero che conta e\' run= della riga [GPU] dim=128: e\' i secondi dell\'argmin sul server.')

## Cosa guardare
- `run=` della riga **`[GPU] ... dim=128`** = i **secondi dell'argmin cifrato sul server** (l'obiettivo "2-3 secondi").
- Confronto con `[CPU] dim=128` = **speedup** GPU/CPU sulla stessa macchina.
- `bit=` deve essere ~11 a dim 128 (stesso punto operativo del caso reale a ~90% accuratezza).

Copia tutto l'output e incollamelo: lo metto nei findings.